In [3]:
!pip install jax flax optax torchvision
!pip install "numpy<2"

/Users/soonbeom/anaconda3/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (114 kB)
Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl (14.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
numba 0.57.1 requires numpy<1.25,>=1.21, but you have numpy 1.26.4 which is incompatible.
tensorflow-macos 2.14.0 requires ml-dtypes==0.2.0, but you have ml-dtypes 0.5.4 which is incompatible.
jax 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which 

In [4]:
import jax
import jax.numpy as jnp
from jax import random, grad, jit
import flax.linen as nn
import optax
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np


# ---------- Hyperparameters ----------
learning_rate = 1e-3
batch_size = 64
num_epochs = 5
num_classes = 10
seed = 0


# ---------- Model (Flax) ----------
class CNN(nn.Module):
    """MNIST 용 간단한 CNN. PyTorch 버전과 동일 구조."""

    @nn.compact
    def __call__(self, x):
        # 입력: (B, 28, 28, 1)   ← JAX 는 channel last !
        x = nn.Conv(features=6, kernel_size=(3, 3), padding='SAME')(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))   # (B, 14, 14, 6)

        x = nn.Conv(features=16, kernel_size=(3, 3), padding='SAME')(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))   # (B, 7, 7, 16)

        x = x.reshape((x.shape[0], -1))       # flatten → (B, 784)
        x = nn.Dense(features=num_classes)(x) # → (B, 10)
        return x


# ---------- Data loading ----------
def numpy_collate(batch):
    """torch Tensor 를 numpy 배열로 변환하면서 배치화."""
    imgs, labels = zip(*batch)
    imgs = np.stack([np.array(img) for img in imgs])     # (B, 1, 28, 28)
    imgs = imgs.transpose(0, 2, 3, 1)                    # → (B, 28, 28, 1)
    labels = np.array(labels)
    return imgs, labels


transform = transforms.ToTensor()
train_ds = datasets.MNIST('dataset/', train=True, download=True, transform=transform)
test_ds  = datasets.MNIST('dataset/', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          collate_fn=numpy_collate, drop_last=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                          collate_fn=numpy_collate)


# ---------- Init model & optimizer ----------
key = random.PRNGKey(seed)
model = CNN()

# 모델 파라미터 초기화 (dummy 입력으로 shape 추론)
dummy_input = jnp.ones((1, 28, 28, 1))
params = model.init(key, dummy_input)['params']

optimizer = optax.adam(learning_rate)
opt_state = optimizer.init(params)


# ---------- Loss & metrics ----------
def loss_fn(params, x, y):
    logits = model.apply({'params': params}, x)
    # cross-entropy with integer labels
    one_hot = jax.nn.one_hot(y, num_classes)
    loss = -jnp.mean(jnp.sum(one_hot * jax.nn.log_softmax(logits), axis=-1))
    return loss, logits


# ---------- Train step (JIT 컴파일) ----------
@jit
def train_step(params, opt_state, x, y):
    (loss, logits), grads = jax.value_and_grad(loss_fn, has_aux=True)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
    return params, opt_state, loss, acc


@jit
def eval_step(params, x, y):
    logits = model.apply({'params': params}, x)
    acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
    return acc


# ---------- Training loop ----------
for epoch in range(num_epochs):
    running_loss, running_acc, n = 0.0, 0.0, 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x = jnp.asarray(x)
        y = jnp.asarray(y)

        params, opt_state, loss, acc = train_step(params, opt_state, x, y)

        running_loss += float(loss)
        running_acc  += float(acc)
        n += 1

        if batch_idx % 200 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}] '
                  f'Batch [{batch_idx}/{len(train_loader)}] '
                  f'Loss: {float(loss):.4f} Acc: {float(acc)*100:.2f}%')

    print(f'===> Epoch {epoch+1} '
          f'avg loss: {running_loss/n:.4f}  avg acc: {running_acc/n*100:.2f}%')


# ---------- Evaluation ----------
def check_accuracy(loader, params, split='test'):
    print(f'Checking accuracy on {split} data')
    correct, total = 0, 0
    for x, y in loader:
        x = jnp.asarray(x)
        y = jnp.asarray(y)
        acc = eval_step(params, x, y)
        correct += float(acc) * len(y)
        total += len(y)
    print(f'Got {int(correct)}/{total} with accuracy {correct/total*100:.2f}%')


check_accuracy(train_loader, params, split='train')
check_accuracy(test_loader,  params, split='test')


RuntimeError: Numpy is not available